In [3]:
import sys
sys.path.append('../src/')
import numpy as np
from phoenix_models import PhoenixInterpGrid
from convolution import gauss_convolve
from astropy.io import fits
from astropy import units as u
from astropy.constants import h, c
from scipy.signal import argrelextrema
import argparse
import itertools
from tqdm import tqdm

orders = [1,2,3,5]
for o in orders:
    print(o,flush=True)
    ## SPIRou Wavelength Grid ##########
    e = fits.open('../data/SPIRou_wavelength_solution.fits')
    # Collect data
    a = e[1].data
    # Currently using Order 20: 1280-1320 nm
    wave_data =  a[o]/1000 #Convert nm to microns

    ## PHOENIX Inputs ##############
    phnx_inputs = dict()

    # Parameters (range or fixed parameter)
    phnx_inputs['teff'] = [2300,4000]
    phnx_inputs['logg'] = [0,5.5]
    phnx_inputs['metal'] = [-4,1]
    phnx_inputs['alpha'] = 0.0  # Constant so not interpolated (can be if needed)

    #
    # Other parameters
    #

    # wavelenngth range (from data to be fitted)
    wv_range = [np.min(wave_data), np.max(wave_data)]  # in microns

    phnx_inputs['wv_range'] = wv_range  # full range: [0.955, 2.516]
    phnx_inputs['resolution'] = 200_000  # Resolution of the output
    phnx_inputs['oversampling'] = 3  # Oversampling of the grid used for interpolation
    phnx_inputs['n_fwhm'] = 7  # Where to cut the convolution kernel
    phnx_inputs['output_wv_grid'] = None  # The grid used for interpolation can be directly specified
    phnx_inputs['method'] = 'linear'   # Type of interpolation (more than linear might be too heavy for MCMC)
    phnx_inputs['query'] = True  # Access to internet? (will download missing PHOENIX models)
    print("Defined inputs",flush=True)
    ## Create the spectra: 
    phoenix_spec = PhoenixInterpGrid(**phnx_inputs)
    print("Got spectra",flush=True)
    ''' Dont do the following because it adds tilt which I dont want in the normalized data'''
    # ### Convert to right units
    # # Convert wavelength grid to nm
    # Lambda = phoenix_spec.wgrid * 1000 * u.nm  # shape = (n_lambda,)
    # energy_per_photon = (h * c / Lambda).to(u.erg) / u.ph  # shape = (n_lambda,)

    # # Apply row-wise conversion
    # def convert_to_photon_flux(flux_row):
    #     # flux_row is in erg/s/cm²/cm
    #     flux_density_erg = (flux_row * u.erg / (u.s * u.cm**2 * u.cm)).to(
    #         u.erg / (u.s * u.cm**2 * u.nm)
    #     )
    #     return (flux_density_erg / energy_per_photon).to(u.ph / (u.s * u.cm**2 * u.nm)).value

    # # Apply the conversion to each row in the column
    # phoenix_spec.flux['Spectrum'] = phoenix_spec.flux['Spectrum'].apply(convert_to_photon_flux)

    ## This output spectrum from PHOENIX has size 18681. This will need to be padded to train on a NN

    # Normalize 
    phoenix_spec.flux['Normalization_Factor'] = phoenix_spec.flux['Spectrum'].apply(np.median)
    phoenix_spec.flux['Normalized'] = phoenix_spec.flux['Spectrum']/phoenix_spec.flux['Normalization_Factor']
    #
    # First we broaden the spectrum to fit SPIRou broadening
    #
    SPIRou_broadening = 70_000
    phoenix_spec.flux['Broadened'] = phoenix_spec.flux['Normalized'].apply(
                            lambda x: gauss_convolve(phoenix_spec.wgrid, x, SPIRou_broadening,
                            n_fwhm=7, res_rtol=1e-6, mode='same',i_plot=False)[1])



    #
    # Finally we pad it so that it's size so that it can be divisible by 256
    #
    def pad_array(arr, target_size=18_688, pad_value=1):
        pad_total = target_size - len(arr)
        pad_left = pad_total // 2
        pad_right = pad_total - pad_left  # Ensures total padding adds up correctly
        return np.pad(arr, (pad_left, pad_right), constant_values=pad_value)

    length = len(phoenix_spec.flux['Broadened'].iloc[0])
    ts = ((length + 31) // 32) * 32
    print("About to do final",flush=True)
    phoenix_spec.flux['Final'] = phoenix_spec.flux['Broadened'].apply(lambda x: pad_array(x,target_size=ts))
    phoenix_spec.flux['Wavelength'] = [phoenix_spec.wgrid*1000]*len(phoenix_spec.flux) # convert from micron to nm
    phoenix_spec.flux['Wavelength'] = phoenix_spec.flux['Wavelength'].apply(lambda x: pad_array(x,target_size=ts))

    #
    # Now let's create the datasets
    #

    # Step 1: Shuffle the DataFrame
    df_shuffled = phoenix_spec.flux.sample(frac=1, random_state=42)  # Shuffle with a fixed random state for reproducibility

    # Step 2: Split into Training and Validation Sets
    train_ratio = 0.8  # 80% for training, 20% for validation
    train_size = int(train_ratio * len(df_shuffled))
    print(train_size,flush=True)
    train_data = df_shuffled[:train_size]
    validation_data = df_shuffled[train_size:]

    # Save
    n = str(o)
    if o<10:
        n = "0"+str(o)
    print("About to save",flush=True)
    train_data.to_pickle("../data/SPIRou"+n+"_train.df")
    validation_data.to_pickle("../data/SPIRou"+n+"_val.df")


1
Defined inputs
{'teff': 2300, 'logg': 0.0, 'metal': -4.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.0, 'metal': -3.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.0, 'metal': -2.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.0, 'metal': -1.5, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.0, 'metal': -1.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.0, 'metal': -0.5, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.0, 'metal': 0.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.0, 'metal': 0.5, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.0, 'metal': 1.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.5, 'metal': -4.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.5, 'metal': -3.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.5, 'metal': -2.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.5, 'metal': -1.5, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.5, 'metal': -1.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.5, 'metal': -0.5, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.5, 'metal': 0.0, 'alpha': 0.0}
{'teff': 2300, 'logg': 0.5, 'metal': 0.5, 'alpha': 0.0}
{'teff': 2300, 'log

In [ ]:
cd 